In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
cd drive/My\ Drive/"Path to your folder" # Path to your folder 부분 입력

[Errno 2] No such file or directory: 'drive/My Drive/RND/NIA'
/content/drive/My Drive/RND/NIA


In [ ]:
disease = "anxiety" # addiction, depression, anxiety 중 택 1

In [ ]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

# CustomBertForSequenceRegression 클래스 정의
class CustomBertForSequenceRegression(BertForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = 1  # 레이블 수 지정
        self.regressor = nn.Linear(config.hidden_size, self.num_labels)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, position_ids=None, head_mask=None, inputs_embeds=None, labels=None, output_attentions=None, output_hidden_states=None, return_dict=None):
        outputs = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        sequence_output = outputs[0]
        pooled_output = sequence_output[:, 0, :]  # 첫 번째 토큰의 출력 사용
        logits = self.regressor(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = nn.MSELoss()
            loss = loss_fct(logits, labels)
        return (loss, logits) if loss is not None else logits

# 데이터셋 토큰화 함수 정의
def tokenize_function(examples):
    return tokenizer(examples['input'], padding='max_length', truncation=True)

# MultiLabelDataCollator 정의
class MultiLabelDataCollator:
    def __call__(self, features):
        batch = {}
        batch['input_ids'] = torch.stack([f['input_ids'] for f in features])
        batch['attention_mask'] = torch.stack([f['attention_mask'] for f in features])
        if 'token_type_ids' in features[0]:
            batch['token_type_ids'] = torch.stack([f['token_type_ids'] for f in features])
        batch['labels'] = torch.stack([torch.tensor(f['label'], dtype=torch.float) for f in features])
        return batch

# CustomTrainer 정의
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs[1] if isinstance(outputs, tuple) else outputs
        loss_fct = nn.MSELoss()
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# 0과 3사이 가장 가까운 정수로 변환하는 함수 정의
def closest_integer(predictions):
    return min(max(round(predictions), 0), 3)

# 예측 함수
def predict(sentence, model, tokenizer):
    inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}  # GPU로 이동
    model.to(device)  # 모델을 GPU로 이동
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs[1] if isinstance(outputs, tuple) else outputs
    prediction = logits.squeeze().tolist()  # 텐서를 리스트로 변환
    prediction = closest_integer(prediction)  # 가장 가까운 정수로 변환
    return prediction


In [ ]:
import os
import json

# TEST SET 폴더 입장
folder_path = './data/test' # 폴더 내 test 데이터셋이 있는 경로 설정

# 폴더 내 disease 관련 JSON 파일 리스트 가져오기
test_json_files = [f for f in os.listdir(folder_path) if f.endswith('.json') and (disease in f or 'normal' in f)]

# JSON 파일 하나씩 열어서 데이터 가져오기
test_labels = []
test_txts = []

for json_file in test_json_files:
    file_path = os.path.join(folder_path, json_file)

    with open(file_path, 'r', encoding='utf-8') as f:
        js = json.load(f)
        test_labels.append(js.get(disease, None))
        paragraghs = js.get("paragraph", None)

        sentences = ""
        for token in paragraghs:
            speaker = token.get("paragraph_speaker", "")
            text = token.get("paragraph_text", "")

            # 문장 생성 (스피커와 텍스트를 합쳐서 저장)
            token_sentence = f"{speaker}: {text}\n"
            sentences += token_sentence  # 문장을 계속해서 더함

        test_txts.append(sentences)

In [ ]:
if __name__ == "__main__":

    # GPU 또는 CPU 장치 설정
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 모델을 불러와서 예측 수행하기
    loaded_model = CustomBertForSequenceRegression.from_pretrained("./trained_model_kluebert_"+disease).to(device)
    loaded_tokenizer = BertTokenizer.from_pretrained("./trained_model_kluebert_"+disease)

    loaded_model.eval()

    # 예측 예시 (로드된 모델을 사용),  test_sentence 에는 임의의 텍스트들을 편집 가능하게 넣고 테스트해보실 수 있습니다.
    test_sentence = "내가 가는 이 길이 어디로 가는지 어디로 날 데려가는지 그곳은 어딘지 알 수 없지만 오늘도 난 걸어가고 있네." # 테스트할 문장 입력
    predicted_numbers = predict(test_sentence, loaded_model, loaded_tokenizer)
    print(f"\nInput: {test_sentence}\nPredicted numbers with loaded model: {predicted_numbers}")



Input: 내가 가는 이 길이 어디로 가는지 어디로 날 데려가는지 그곳은 어딘지 알 수 없지만 오늘도 난 걸어가고 있네 사람들은 길이 다 정해져 있는지 아니면 자기가 자신의 길을 만들어 가는지 알 수 없지만 이렇게 또 걸어가고 있네
Predicted numbers with loaded model: 1


In [ ]:
import pandas as pd

assert len(test_labels) == len(test_txts)

test_df = pd.DataFrame({
    'filename': test_json_files,
    'input': test_txts,
    'original_label': test_labels,
    'original_label_zeroone' : [0 if x == 0 else 1 for x in test_labels]
})

# 예측 결과를 담을 리스트
predicted_labels = []

# 각 문장에 대해 예측 수행
for sentence in test_df['input']:
    predicted_numbers = predict(sentence, loaded_model, loaded_tokenizer)
    predicted_labels.append(predicted_numbers)

# 예측 결과를 새로운 컬럼 'predicted_label'에 추가
test_df['predicted_label'] = predicted_labels
test_df['predicted_label_zeroone'] = [0 if x == 0 else 1 for x in predicted_labels]


68
68


In [ ]:
# numpy 배열로 변환 (계산을 쉽게 하기 위해)
original_label_zeroone = np.array(test_df['original_label_zeroone'])
predicted_label_zeroone = np.array(test_df['predicted_label_zeroone'])

# 정확도 계산: 예측이 정확한 경우의 수 / 전체 데이터 수
accuracy_zeroone = np.mean(original_label_zeroone == predicted_label_zeroone)

# 정확도 출력
print(disease, "0~1")
print(f"Accuracy: {accuracy_zeroone * 100:.2f}%")

anxiety 0~1
Accuracy: 73.53%
